# Final Model Evaluation on Test Set - Google Colab Version

This notebook is adapted to run on Google Colab. It evaluates the best models from each approach:
1. AutoGluon (Hybrid Model)
2. Naive Baseline

**IMPORTANT:** Before running, make sure to set the `project_root` variable in the first code cell to the correct path of your project directory in Google Drive.

In [1]:
# === 1. SETUP FOR GOOGLE COLAB ===

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Install dependencies
# Note: This may take a few minutes.
!pip install autogluon

# --- IMPORTANT: SET YOUR PROJECT PATH ---
# Replace '/content/drive/My Drive/path/to/your/project' with the actual path to the 'Final_Project' directory on your Google Drive.
# Example: '/content/drive/My Drive/Colab Notebooks/Final_Project'
project_root = '/content/drive/My Drive/FS_MADS_DL_2025/final_project' # <-- CHANGE THIS

import os
import sys

# Add project root to Python path
if project_root not in sys.path:
    sys.path.append(project_root)

# Change working directory to project root
try:
    os.chdir(project_root)
    print(f"Working directory successfully changed to: {os.getcwd()}")
except FileNotFoundError:
    print(f"ERROR: The specified project_root '{project_root}' does not exist. Please update the path.")

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.5/259.5 kB 27.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of openxlab to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of openxlab to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pyp

In [2]:
import os
import sys
import json
import glob
import time
import pandas as pd
import numpy as np
import torch
import random
from tqdm.auto import tqdm
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor
import logging

# Configure logging
logging.basicConfig(level=logging.INFO, format='[%(levelname)s] %(message)s')

# The project_root and sys.path are now handled by the Colab setup cell above.
from src.datamodule import ElectricityDataModule

## 1. Configuration

In [3]:
# The 'project_root' variable is defined in the Colab setup cell.
BASE_DIR = project_root
DATA_DIR = os.path.join(BASE_DIR, "data")
TEST_DIR = os.path.join(DATA_DIR, "test")
SCALERS_DIR = os.path.join(DATA_DIR, "scalers")
MODELS_DIR = os.path.join(BASE_DIR, "models")
RESULTS_DIR = os.path.join(BASE_DIR, "results")

os.makedirs(RESULTS_DIR, exist_ok=True)

TARGET_COLS = ["high", "low", "close", "volume"]
INPUT_CHUNK_LENGTH = 48
OUTPUT_CHUNK_LENGTH = 10
SEED = 827
MAX_TEST_FILES = 25  # Set None to load all files
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Define covariates - these will be returned by the dataloader
# Market features (historical price/volume patterns)
PAST_COVARIATES = [
    'close_lag_adj_1', 'close_delta_adj_1', 'volume_adj_1',
    'close_lag_adj_2', 'close_delta_adj_2', 'volume_adj_2',
    'close_lag_adj_3', 'close_delta_adj_3', 'volume_adj_3',
    'close_lag_adj_4', 'close_delta_adj_4', 'volume_adj_4',
    'close_lag_adj_5', 'close_delta_adj_5', 'volume_adj_5',
    'close_lag_adj_6', 'close_delta_adj_6', 'volume_adj_6',
    'nearest_liquid_contract_close', 'cross_contract_mean',
    'is_trading'
]

# Time features (same for past and future windows)
TIME_COVARIATES = [
    'time_to_delivery', 'hour_of_day', 'day_of_week',
    'week_of_year', 'month', 'is_weekend',
]

# Regressor covariates (exclude is_trading since it's classifier's target)
REGRESSOR_PAST_COVARIATES = [c for c in PAST_COVARIATES if c != 'is_trading']

print('='*80)
print(f"Using {MAX_TEST_FILES} test files")
print('='*80)

test_files = sorted(glob.glob(os.path.join(TEST_DIR, "*.parquet")))
random.seed(SEED)
random.shuffle(test_files)

if MAX_TEST_FILES is not None:
    test_files = test_files[:MAX_TEST_FILES]
    print('='*80)
    print(f"Using {MAX_TEST_FILES} test files")
    print('='*80)

# Extract asset names (remove .parquet extension) for dataloader
selected_assets = [os.path.splitext(os.path.basename(f))[0] for f in test_files]
print(f"Selected assets: {selected_assets}")

print('='*80)
print("Configuration loaded successfully")
print(f"Test data directory: {TEST_DIR}")
print(f"Number of test files: {len(test_files)}")
print(f"Device: {device}")
print(f"Past market covariates: {len(PAST_COVARIATES)}")
print(f"Time covariates: {len(TIME_COVARIATES)}")
print('='*80)

Using 25 test files
Using 25 test files
Selected assets: ['Thu20Q1', 'Wed13Q2', 'Fri18Q3', 'Sun13Q4', 'Tue03Q1', 'Wed13Q3', 'Mon19Q4', 'Wed06Q1', 'Mon05Q4', 'Thu05Q3', 'Sat09Q3', 'Tue18Q3', 'Fri09Q4', 'Mon16Q1', 'Mon05Q2', 'Tue15Q3', 'Fri20Q1', 'Sat03Q1', 'Tue15Q2', 'Sat13Q4', 'Fri18Q4', 'Mon08Q1', 'Fri20Q2', 'Wed19Q4', 'Tue02Q4']
Configuration loaded successfully
Test data directory: /content/drive/My Drive/FS_MADS_DL_2025/final_project/data/test
Number of test files: 25
Device: cuda
Past market covariates: 21
Time covariates: 6


In [4]:
def simple_smape(y_true, y_pred, eps=1e-6):
    num = torch.abs(y_true - y_pred)
    den = torch.abs(y_true) + torch.abs(y_pred) + eps
    smape = 2.0 * num / den
    return smape.mean()

## 2. Evaluate Naive Baseline

In [5]:
print('='*80)
print("EVALUATING NAIVE BASELINE")
print('='*80)

start = time.time()

# Create datamodule WITHOUT covariates for naive baseline (doesn't need them)
datamodule = ElectricityDataModule(
    train_parquet=os.path.join(DATA_DIR, "train"),
    val_parquet=os.path.join(DATA_DIR, "val"),
    test_parquet=TEST_DIR,
    scalers_dir=SCALERS_DIR,
    batch_size=32,
    num_workers=2, # Recommended to use 2 workers in Colab
    dataset_kwargs={
        'stride': OUTPUT_CHUNK_LENGTH,  # Non-overlapping windows
        'assets_list': selected_assets,
    }
)
datamodule.setup(stage='test')
test_dataloader = datamodule.test_dataloader()

all_naive_forecasts = []
all_ground_truths = []

for batch in tqdm(test_dataloader, desc="Naive forecasting"):
    past, _, future, _, _, _ = batch
    last_values = past[:, -1:, :]
    naive_pred = last_values.repeat(1, OUTPUT_CHUNK_LENGTH, 1)
    all_naive_forecasts.append(naive_pred)
    all_ground_truths.append(future)

naive_forecasts = torch.cat(all_naive_forecasts, dim=0).to(device)
naive_ground_truths = torch.cat(all_ground_truths, dim=0).to(device)

naive_smape = simple_smape(naive_ground_truths, naive_forecasts).item() * 100
naive_by_target = {}
for i, target_name in enumerate(TARGET_COLS):
    target_smape = simple_smape(naive_ground_truths[:, :, i], naive_forecasts[:, :, i]).item() * 100
    naive_by_target[target_name] = target_smape

print(f"Evaluated based on {MAX_TEST_FILES if MAX_TEST_FILES is not None else 'all'} files in {((time.time() - start)/60):.4f} minutes")
print(f"Naive Baseline sMAPE: {naive_smape:.2f}%")
for target_name, smape in naive_by_target.items():
    print(f"  {target_name:8s}: {smape:.2f}%")
print('='*80)

EVALUATING NAIVE BASELINE


Naive forecasting: 0it [00:00, ?it/s]

Evaluated based on 25 files in 1.0131 minutes
Naive Baseline sMAPE: 7.16%
  high    : 4.69%
  low     : 4.71%
  close   : 4.71%
  volume  : 14.54%


## 3. Evaluate AutoGluon Hybrid Model

In [6]:
# Create a datamodule WITH ALL covariates pre-computed by dataloader
print("Creating dataloader with PRE-COMPUTED covariates for AutoGluon...")
datamodule_with_covariates = ElectricityDataModule(
    train_parquet=os.path.join(DATA_DIR, "train"),
    val_parquet=os.path.join(DATA_DIR, "val"),
    test_parquet=TEST_DIR,
    scalers_dir=SCALERS_DIR,
    batch_size=32,
    num_workers=2, # Recommended to use 2 workers in Colab
    dataset_kwargs={
        'return_covariates': True,
        'past_covariate_cols': PAST_COVARIATES,  # Market features for past window
        'past_time_covariate_cols': TIME_COVARIATES,  # Time features for past window (NEW!)
        'future_covariate_cols': TIME_COVARIATES,  # Time features for future window
        'stride': OUTPUT_CHUNK_LENGTH,  # Non-overlapping windows
        'assets_list': selected_assets,
    }
)
datamodule_with_covariates.setup(stage='test')
test_dataloader_with_covariates = datamodule_with_covariates.test_dataloader()
print("✓ Created dataloader with ALL pre-computed covariates")
print(f"  Input chunk length: {INPUT_CHUNK_LENGTH}")
print(f"  Output chunk length: {OUTPUT_CHUNK_LENGTH}")
print(f"  Stride: {OUTPUT_CHUNK_LENGTH} (non-overlapping)")
print(f"  Past market features: {len(PAST_COVARIATES)}")
print(f"  Past time features: {len(TIME_COVARIATES)} (pre-computed!)")
print(f"  Future time features: {len(TIME_COVARIATES)} (pre-computed!)")

Creating dataloader with PRE-COMPUTED covariates for AutoGluon...
✓ Created dataloader with ALL pre-computed covariates
  Input chunk length: 48
  Output chunk length: 10
  Stride: 10 (non-overlapping)
  Past market features: 21
  Past time features: 6 (pre-computed!)
  Future time features: 6 (pre-computed!)


In [7]:
print("="*80)
print("EVALUATING AUTOGLUON HYBRID MODEL")
print("="*80)

start = time.time()

AUTOGLUON_CLASSIFIER_DIR = os.path.join(MODELS_DIR, "autogluon_trading_classifier")
AUTOGLUON_REGRESSOR_DIR = os.path.join(MODELS_DIR, "autogluon_hybrid_regressor")

autogluon_hybrid_smape = float('inf')
autogluon_hybrid_by_target = {}

if os.path.exists(AUTOGLUON_CLASSIFIER_DIR) and os.path.exists(AUTOGLUON_REGRESSOR_DIR):
    try:
        print(f"Loading AutoGluon hybrid models...")
        classifier_predictor = TimeSeriesPredictor.load(AUTOGLUON_CLASSIFIER_DIR)
        regressor_predictor = TimeSeriesPredictor.load(AUTOGLUON_REGRESSOR_DIR)
        print(f"✓ Loaded classifier and regressor predictors.")

        all_hybrid_forecasts = []
        all_hybrid_gts = []

        # Find is_trading index in past covariates
        is_trading_idx = PAST_COVARIATES.index('is_trading')

        # Create covariate name to index mappings for fast lookup
        past_cov_idx_map = {name: idx for idx, name in enumerate(PAST_COVARIATES)}
        time_cov_idx_map = {name: idx for idx, name in enumerate(TIME_COVARIATES)}

        for batch_idx, batch in enumerate(tqdm(test_dataloader_with_covariates, desc="AutoGluon Hybrid")):
            # Unpack 9 elements
            past, past_mask, future, future_mask, asset_ids, past_ts_list, past_covariates, past_time_covariates, future_covariates = batch
            batch_size = past.shape[0]

            # Build DataFrames using vectorized operations and pre-computed features
            clf_data = []
            reg_data = []
            known_cov_data = []

            for i in range(batch_size):
                item_id_base = f"window_{i}"

                # Use REAL timestamps from dataloader
                real_timestamps = pd.to_datetime(past_ts_list[i])
                last_past_ts = real_timestamps[-1]
                future_timestamps = pd.date_range(
                    start=last_past_ts + pd.Timedelta(minutes=15),
                    periods=OUTPUT_CHUNK_LENGTH,
                    freq='15min'
                )

                # Extract numpy arrays for this sample
                past_targets = past[i].cpu().numpy()  # [INPUT_CHUNK_LENGTH, 4]
                past_covs = past_covariates[i].cpu().numpy()  # [INPUT_CHUNK_LENGTH, len(PAST_COVARIATES)]
                past_time_covs = past_time_covariates[i].cpu().numpy()  # [INPUT_CHUNK_LENGTH, len(TIME_COVARIATES)] - PRE-COMPUTED!
                future_covs = future_covariates[i].cpu().numpy()  # [OUTPUT_CHUNK_LENGTH, len(TIME_COVARIATES)] - PRE-COMPUTED!

                # --- BUILD CLASSIFIER INPUT DATA ---
                clf_df = pd.DataFrame({
                    'item_id': item_id_base,
                    'timestamp': real_timestamps,  # REAL timestamps!
                    'is_trading': past_covs[:, is_trading_idx],
                    'high': past_targets[:, 0],
                    'low': past_targets[:, 1],
                    'close': past_targets[:, 2],
                    'volume': past_targets[:, 3],
                    'block': 0.0
                })

                # Add past market covariates (exclude is_trading, already added)
                for cov_name in PAST_COVARIATES:
                    if cov_name != 'is_trading':
                        clf_df[cov_name] = past_covs[:, past_cov_idx_map[cov_name]]

                # Use pre-computed time features directly!
                for cov_name in TIME_COVARIATES:
                    clf_df[cov_name] = past_time_covs[:, time_cov_idx_map[cov_name]]

                clf_data.append(clf_df)

                # --- BUILD REGRESSOR INPUT DATA ---
                for j, col_name in enumerate(TARGET_COLS):
                    reg_item_id = f"{item_id_base}_{col_name}"

                    reg_df = pd.DataFrame({
                        'item_id': reg_item_id,
                        'timestamp': real_timestamps,  # REAL timestamps!
                        'target': past_targets[:, j],
                        'block': 0.0
                    })

                    # Add past market covariates (exclude is_trading)
                    for cov_name in REGRESSOR_PAST_COVARIATES:
                        reg_df[cov_name] = past_covs[:, past_cov_idx_map[cov_name]]

                    # Use pre-computed time features directly!
                    for cov_name in TIME_COVARIATES:
                        reg_df[cov_name] = past_time_covs[:, time_cov_idx_map[cov_name]]

                    reg_data.append(reg_df)

                # --- BUILD KNOWN COVARIATES FOR FUTURE ---
                # Use pre-computed future covariates directly!

                # Classifier known covariates
                clf_known_cov = pd.DataFrame({
                    'item_id': item_id_base,
                    'timestamp': future_timestamps,
                })
                for cov_name in TIME_COVARIATES:
                    clf_known_cov[cov_name] = future_covs[:, time_cov_idx_map[cov_name]]
                known_cov_data.append(clf_known_cov)

                # Regressor known covariates (one per target)
                for j, col_name in enumerate(TARGET_COLS):
                    reg_item_id = f"{item_id_base}_{col_name}"
                    reg_known_cov = pd.DataFrame({
                        'item_id': reg_item_id,
                        'timestamp': future_timestamps,
                    })
                    for cov_name in TIME_COVARIATES:
                        reg_known_cov[cov_name] = future_covs[:, time_cov_idx_map[cov_name]]
                    known_cov_data.append(reg_known_cov)

            # Concatenate all DataFrames for this batch
            clf_full_df = pd.concat(clf_data, ignore_index=True)
            reg_full_df = pd.concat(reg_data, ignore_index=True)
            known_cov_full_df = pd.concat(known_cov_data, ignore_index=True)

            # Convert to TimeSeriesDataFrame
            classifier_ts = TimeSeriesDataFrame.from_data_frame(clf_full_df, id_column='item_id', timestamp_column='timestamp')
            regressor_ts = TimeSeriesDataFrame.from_data_frame(reg_full_df, id_column='item_id', timestamp_column='timestamp')
            known_cov_ts = TimeSeriesDataFrame.from_data_frame(known_cov_full_df, id_column='item_id', timestamp_column='timestamp')

            # Get predictions
            is_trading_forecasts = classifier_predictor.predict(classifier_ts, model="DeepAR", known_covariates=known_cov_ts)
            price_volume_forecasts = regressor_predictor.predict(regressor_ts, model="TemporalFusionTransformer_FineTuned", known_covariates=known_cov_ts)

            # Combine predictions: final = classifier × regressor
            batch_final_preds = torch.zeros_like(future)

            for i in range(batch_size):
                item_id_base = f"window_{i}"

                # Get classifier predictions (binary: 0 or 1 for is_trading)
                trading_preds = is_trading_forecasts.loc[item_id_base]['mean'].values
                trading_mask = (trading_preds >= 0.5).astype(float)  # Convert to 0/1

                # Get regressor predictions for each target
                for j, col_name in enumerate(TARGET_COLS):
                    regressor_item_id = f"{item_id_base}_{col_name}"
                    if regressor_item_id in price_volume_forecasts.item_ids:
                        reg_values = price_volume_forecasts.loc[regressor_item_id]['mean'].values
                        # Hybrid prediction: multiply by trading mask
                        batch_final_preds[i, :, j] = torch.from_numpy(reg_values * trading_mask).float()

            all_hybrid_forecasts.append(batch_final_preds)
            all_hybrid_gts.append(future)

        hybrid_forecasts = torch.cat(all_hybrid_forecasts, dim=0).to(device)
        hybrid_gts = torch.cat(all_hybrid_gts, dim=0).to(device)

        autogluon_hybrid_smape = simple_smape(hybrid_gts, hybrid_forecasts).item() * 100
        autogluon_hybrid_by_target = {}
        for i, target_name in enumerate(TARGET_COLS):
            autogluon_hybrid_by_target[target_name] = simple_smape(hybrid_gts[:, :, i], hybrid_forecasts[:, :, i]).item() * 100

        print(f"Evaluated based on {MAX_TEST_FILES if MAX_TEST_FILES is not None else 'all'} files in {((time.time() - start)/60):.4f} minutes")
        print(f"\nAutoGluon Hybrid Model sMAPE: {autogluon_hybrid_smape:.2f}%")
        for target_name, smape in autogluon_hybrid_by_target.items():
            print(f"  {target_name:8s}: {smape:.2f}%")

    except Exception as e:
        print(f"✗ Error evaluating AutoGluon hybrid model: {e}")
        import traceback
        traceback.print_exc()
else:
    print(f"✗ AutoGluon hybrid models not found. Run the training notebook first.")

print("="*80)

EVALUATING AUTOGLUON HYBRID MODEL
Loading AutoGluon hybrid models...
✓ Loaded classifier and regressor predictors.


AutoGluon Hybrid: 0it [00:00, ?it/s]

Evaluated based on 25 files in 150.5762 minutes

AutoGluon Hybrid Model sMAPE: 21.36%
  high    : 23.40%
  low     : 23.42%
  close   : 23.41%
  volume  : 15.23%


## 5. Final Comparison and Results

In [8]:
logging.info("--- Final Results ---")
final_results = {
    'AutoGluon Hybrid': autogluon_hybrid_smape,
    'Naive Baseline': naive_smape,
}
sorted_results = sorted(final_results.items(), key=lambda x: x[1])

print("Overall sMAPE (ranked):")
for rank, (model, smape) in enumerate(sorted_results, 1):
    if smape != float('inf'):
        print(f"  {rank}. {model:25s}: {smape:.2f}%")
    else:
        print(f"  {rank}. {model:25s}: N/A")

best_model_name, best_smape = sorted_results[0]
print(f"\nBEST MODEL: {best_model_name} with sMAPE: {best_smape:.2f}%")

Overall sMAPE (ranked):
  1. Naive Baseline           : 7.16%
  2. AutoGluon Hybrid         : 21.36%

BEST MODEL: Naive Baseline with sMAPE: 7.16%
